# MLOps Assignment 3
**Ausaf Ahmad | MTech AI/ML | IIT Jodhpur | June 2026**

Zero-shot disaster tweet classification using **GLiClass LLaMA 1.3B** with:
- Hugging Face model download
- Kaggle NLP Disaster Tweets competition
- Weights & Biases experiment tracking
- Docker deployment

---
## Useful Links
| Service | Link |
|---|---|
| HF Model | https://huggingface.co/knowledgator/gliclass-llama-1.3B-v1.0 |
| W&B Project | https://wandb.ai/ausaf-ahmad-iitj3-indian-institute-oftechnoloy-jodhpur/mlops-assignment3 |
| Kaggle Competition | https://www.kaggle.com/competitions/nlp-getting-started |
| My Kaggle Submissions | https://www.kaggle.com/competitions/nlp-getting-started/submissions |

---
## Section 1 — Environment Setup
Run these shell commands once to prepare the environment.

In [ ]:
# 1.1 Install Docker on Fedora WSL (run once in terminal, not needed inside notebook)
# Kept here for reference — execute manually in WSL terminal if Docker is not installed
'''
sudo dnf remove -y docker docker-client docker-common
sudo dnf install -y dnf-plugins-core
sudo dnf config-manager --add-repo https://download.docker.com/linux/fedora/docker-ce.repo
sudo dnf install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
sudo systemctl start docker && sudo systemctl enable docker
sudo usermod -aG docker $USER
newgrp docker
docker --version && docker run hello-world
'''
print("Docker install commands are reference-only. Run in WSL terminal.")

In [ ]:
# 1.2 Install all required packages (run in venv)
import subprocess, sys

packages = [
    'huggingface_hub',
    'transformers',
    'torch',
    'gliclass',
    'kaggle',
    'wandb',
    'pandas',
    'flask',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f"{status} {pkg}")

In [ ]:
# 1.3 Verify key imports
import torch
import pandas as pd
import wandb
from huggingface_hub import whoami
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer
import statistics

print(f"✅ torch      : {torch.__version__}")
print(f"✅ pandas     : {pd.__version__}")
print(f"✅ wandb      : {wandb.__version__}")
print(f"✅ transformers: imported")
print(f"✅ gliclass   : imported")
print("\nAll packages imported successfully!")

---
## Section 2 — Hugging Face Setup & Model Download
⚠️ `huggingface-cli` is deprecated — use `hf` command or `login()` in Python.

In [ ]:
import os
from huggingface_hub import login, whoami

# ⚠️ Remove any stale HF_TOKEN env var that would override login
os.environ.pop('HF_TOKEN', None)

# Paste your Hugging Face token (starts with hf_)
HF_TOKEN = 'hf_your_token_here'  # <-- REPLACE with your actual token

login(token=HF_TOKEN)
user = whoami()
print(f"✅ Logged in as: {user['name']}")

In [ ]:
# download_model.py — Download GLiClass model (~2.7 GB) to local drive
from huggingface_hub import snapshot_download

MODEL_DIR = '/mnt/f/models/gliclass-llama-1.3B'  # adjust path if needed

print(f"Downloading model to: {MODEL_DIR}")
print("This will take a few minutes (~2.7 GB)...")

model_path = snapshot_download(
    repo_id='knowledgator/gliclass-llama-1.3B-v1.0',
    local_dir=MODEL_DIR
)

print(f"\n✅ Model downloaded to: {model_path}")
print("\nFiles:")
for f in sorted(os.listdir(MODEL_DIR)):
    print(f"  {f}")

---
## Section 3 — Kaggle Setup

In [ ]:
# 3.1 Create ~/.kaggle/kaggle.json
import os, json, stat

KAGGLE_USERNAME = 'ausafiitj'          # <-- your Kaggle username
KAGGLE_KEY      = 'YOUR_KAGGLE_KEY'    # <-- paste your Kaggle API key

kaggle_dir  = os.path.expanduser('~/.kaggle')
kaggle_json = os.path.join(kaggle_dir, 'kaggle.json')

os.makedirs(kaggle_dir, exist_ok=True)
with open(kaggle_json, 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(kaggle_json, stat.S_IRUSR | stat.S_IWUSR)  # chmod 600

print(f"✅ kaggle.json written to {kaggle_json}")

In [ ]:
# 3.2 Download competition data
# ⚠️ You must join the competition first at:
#    https://www.kaggle.com/competitions/nlp-getting-started
import kaggle, zipfile

DATA_DIR = '/mnt/f/datasets/disaster-tweets'
os.makedirs(DATA_DIR, exist_ok=True)

kaggle.api.authenticate()
print("✅ Kaggle authenticated")

kaggle.api.competition_download_files(
    competition='nlp-getting-started',
    path=DATA_DIR,
    quiet=False
)

# Unzip
zip_path = os.path.join(DATA_DIR, 'nlp-getting-started.zip')
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print(f"✅ Extracted to {DATA_DIR}")

print("\nFiles:")
for f in os.listdir(DATA_DIR):
    print(f"  {f}")

In [ ]:
# Quick data exploration
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")
print(f"\nTarget distribution:")
print(train_df['target'].value_counts())
print(f"\nSample tweets:")
train_df[['text','target']].sample(5, random_state=42)

---
## Section 4 — Weights & Biases Setup
⚠️ API key must be 40+ characters — get yours from https://wandb.ai/authorize

In [ ]:
import wandb

WANDB_API_KEY = 'YOUR_WANDB_API_KEY'  # <-- paste 40-char key from wandb.ai/authorize

wandb.login(key=WANDB_API_KEY, relogin=True)

# Quick connection test
run = wandb.init(project='mlops-assignment3', name='connection-test', mode='online')
print(f"✅ W&B connected | Run ID: {run.id}")
run.finish()
print("Run finished.")

---
## Section 5 — Test All Service Connections
Equivalent to `test_connections.py`

In [ ]:
# test_connections.py — verify HF + Kaggle + W&B
import os

# 1. Hugging Face
os.environ.pop('HF_TOKEN', None)
from huggingface_hub import whoami
user = whoami()
print(f"✅ HF Connected  : {user['name']}")

# 2. Kaggle
import kaggle
kaggle.api.authenticate()
print(f"✅ Kaggle Connected")

# 3. W&B
import wandb
wandb.login()
run = wandb.init(project='mlops-assignment3', name='connection-test', mode='online')
print(f"✅ W&B Connected  : Run ID = {run.id}")
run.finish()

print("\n🎉 All 3 services connected successfully!")

---
## Section 6 — Load GLiClass Model
⚠️ Key fix: `model.float()` — converts BFloat16 → Float32 to avoid RuntimeError

In [ ]:
import torch
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer

MODEL_DIR = '/mnt/f/models/gliclass-llama-1.3B'  # adjust if needed

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

print('Loading model (this may take 1-2 min)...')
model = GLiClassModel.from_pretrained(MODEL_DIR, torch_dtype=torch.float32)
model = model.float()   # ⚠️ Fix: BFloat16 → Float32

pipeline = ZeroShotClassificationPipeline(
    model, tokenizer,
    classification_type='multi-label',
    device='cpu'
)

print('✅ Model loaded and pipeline ready!')

---
## Section 7 — run_model.py v1: Basic Classification
Expected output: `technology:0.3827  sports:0.3613  business:0.1716`

In [ ]:
# run_model.py v1 — Basic classification
text   = 'Apple announces new iPhone with amazing features'
labels = ['technology', 'sports', 'politics', 'business']

result = pipeline(text, labels, threshold=0.1)

print('=== Classification Results ===')
for item in result:
    for pred in item:
        bar = '█' * int(pred['score'] * 40)
        print(f"  {pred['label']:12s}: {pred['score']:.4f}  {bar}")

---
## Section 8 — run_model.py v2: Classification + W&B Logging
Logs scores to W&B dashboard at https://wandb.ai

In [ ]:
# run_model.py v2 — With W&B logging
import wandb

run = wandb.init(
    project='mlops-assignment3',
    name='gliclass-inference',
    config={'model': 'gliclass-llama-1.3B', 'threshold': 0.1, 'device': 'cpu'}
)
print(f'W&B run ID: {run.id}')

text   = 'Apple announces new iPhone with amazing features'
labels = ['technology', 'sports', 'politics', 'business']

result = pipeline(text, labels, threshold=0.1)

print('\n=== Classification Results ===')
scores = {}
for item in result:
    for pred in item:
        print(f"  {pred['label']:12s}: {pred['score']:.4f}")
        scores[pred['label']] = pred['score']

wandb.log(scores)
print(f'\n✅ Logged to W&B: {scores}')
run.finish()

---
## Section 9 — run_model.py v4 & v5: Failed Kaggle Approaches
Documented for learning — both approaches produced degenerate predictions.

In [ ]:
# run_model.py v4 — Kaggle v1: Fixed threshold=0.3 (FAILED)
# Result: ALL 3263 tweets predicted as disaster (target=1)
# Bug: threshold=0.3 is too low — most disaster scores exceed it

import wandb, pandas as pd

DATA_DIR = '/mnt/f/datasets/disaster-tweets'
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

run = wandb.init(project='mlops-assignment3', name='kaggle-disaster-tweets-v1')

labels = ['disaster', 'not disaster']
predictions = []

# Run on first 50 rows to demo the bug quickly
sample = test_df.head(50)

for i, row in sample.iterrows():
    text   = str(row['text'])
    result = pipeline(text, labels, threshold=0.1)
    scores = {pred['label']: pred['score'] for item in result for pred in item}
    disaster_score = scores.get('disaster', 0)
    prediction = 1 if disaster_score > 0.3 else 0  # BUG: threshold too low
    predictions.append(prediction)

    if i % 10 == 0:
        wandb.log({'progress': i, 'disaster_score': disaster_score})

from collections import Counter
print('Prediction distribution (first 50):', Counter(predictions))
print('❌ Bug: almost all predictions = 1 (disaster)')
run.finish()

In [ ]:
# run_model.py v5 — Kaggle v2: Label comparison (FAILED)
# Result: ALL 3263 tweets predicted as not disaster (target=0)
# Bug: 'not disaster' label always scores higher — model is biased toward it

run = wandb.init(project='mlops-assignment3', name='kaggle-disaster-tweets-v2')

labels = ['disaster', 'not disaster']
predictions = []

sample = test_df.head(50)

for i, row in sample.iterrows():
    text = str(row['text'])
    result = pipeline(text, labels, threshold=0.1)
    scores = {pred['label']: pred['score'] for item in result for pred in item}
    disaster_score     = scores.get('disaster', 0)
    not_disaster_score = scores.get('not disaster', 0)
    prediction = 1 if disaster_score > not_disaster_score else 0  # BUG
    predictions.append(prediction)

    if i % 10 == 0:
        wandb.log({'progress': i, 'disaster_score': disaster_score,
                   'not_disaster_score': not_disaster_score})

print('Prediction distribution (first 50):', Counter(predictions))
print('❌ Bug: almost all predictions = 0 (not disaster)')
run.finish()

---
## Section 10 — run_model.py v6: FINAL — Median Threshold ✅
**Key insight:** Collect all disaster scores → compute median → use as dynamic threshold → ~50/50 balanced predictions.

**Kaggle Score: 0.48605 (BEST)**

In [ ]:
# run_model.py v6 FINAL — Median threshold (Kaggle score: 0.48605)
import statistics, wandb, pandas as pd

DATA_DIR = '/mnt/f/datasets/disaster-tweets'
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')

print(f'Test : {len(test_df)} tweets')
print(f'Train distribution:\n{train_df["target"].value_counts()}')

run = wandb.init(
    project='mlops-assignment3',
    name='kaggle-disaster-tweets-v3',
    config={'model': 'gliclass-llama-1.3B', 'strategy': 'median-threshold', 'device': 'cpu'}
)

labels = ['disaster', 'not disaster']

# ── Step 1: Collect all disaster scores ──────────────────────────
print('\nStep 1/3 — Scoring all tweets...')
disaster_scores = []

for i, row in test_df.iterrows():
    text   = str(row['text'])
    result = pipeline(text, labels, threshold=0.01)  # very low threshold to capture all scores
    scores = {pred['label']: pred['score'] for item in result for pred in item}
    disaster_score     = scores.get('disaster', 0)
    not_disaster_score = scores.get('not disaster', 0)
    disaster_scores.append(disaster_score)

    if i % 100 == 0:
        print(f'  {i:4d}/{len(test_df)} | disaster={disaster_score:.3f} | not_disaster={not_disaster_score:.3f}')
        wandb.log({'progress': i,
                   'disaster_score': disaster_score,
                   'not_disaster_score': not_disaster_score})

# ── Step 2: Compute median as dynamic threshold ──────────────────
median_score = statistics.median(disaster_scores)
print(f'\nStep 2/3 — Median disaster score: {median_score:.4f}')

# ── Step 3: Predict using median threshold ───────────────────────
predictions = [1 if s > median_score else 0 for s in disaster_scores]

# ── Step 4: Save submission CSV ──────────────────────────────────
submission = pd.DataFrame({'id': test_df['id'], 'target': predictions})
submission.to_csv(f'{DATA_DIR}/submission.csv', index=False)

print('\nStep 3/3 — Done!')
print('\n=== Final Results ===')
print(submission['target'].value_counts())
# Expected: 1 → 1631, 0 → 1632  (balanced!)

wandb.log({
    'median_threshold':   median_score,
    'total_disaster':     int(sum(predictions)),
    'total_not_disaster': int(len(predictions) - sum(predictions))
})
run.finish()
print(f'\n✅ Submission saved to {DATA_DIR}/submission.csv')

In [ ]:
# Quick sanity check — preview submission
submission = pd.read_csv(f'{DATA_DIR}/submission.csv')
print('Submission preview:')
display(submission.head(10))
print(f'\nShape   : {submission.shape}')
print(f'Counts  :\n{submission["target"].value_counts()}')

---
## Section 11 — run_model.py v7: Descriptive Labels (Latest)
Uses longer, more descriptive labels to give the model better context.

In [ ]:
# run_model.py v7 — Descriptive labels
import statistics, wandb, pandas as pd

DATA_DIR = '/mnt/f/datasets/disaster-tweets'
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

run = wandb.init(
    project='mlops-assignment3',
    name='kaggle-disaster-tweets-v4-better-labels'
)

# More descriptive labels — gives the model better semantic context
labels = [
    'natural disaster emergency crisis fire flood earthquake',
    'normal everyday tweet unrelated to disaster'
]

disaster_scores = []

for i, row in test_df.iterrows():
    text   = str(row['text'])
    result = pipeline(text, labels, threshold=0.01)
    scores = {pred['label']: pred['score'] for item in result for pred in item}
    disaster_scores.append(scores.get(labels[0], 0))

    if i % 100 == 0:
        print(f'  {i:4d}/{len(test_df)} | disaster_score={disaster_scores[-1]:.3f}')
        wandb.log({'progress': i, 'disaster_score': disaster_scores[-1]})

median_score = statistics.median(disaster_scores)
print(f'\nMedian: {median_score:.4f}')

predictions = [1 if s > median_score else 0 for s in disaster_scores]
submission  = pd.DataFrame({'id': test_df['id'], 'target': predictions})
submission.to_csv(f'{DATA_DIR}/submission_v7.csv', index=False)

print(submission['target'].value_counts())
wandb.log({'median_threshold': median_score, 'total_disaster': int(sum(predictions))})
run.finish()
print(f'\n✅ v7 submission saved to {DATA_DIR}/submission_v7.csv')

---
## Section 12 — Submit to Kaggle

In [ ]:
# 3.3 Submit best submission to Kaggle
import kaggle

DATA_DIR = '/mnt/f/datasets/disaster-tweets'

kaggle.api.authenticate()

kaggle.api.competition_submit(
    file_name=f'{DATA_DIR}/submission.csv',
    message='GLiClass zero-shot v3 median threshold',
    competition='nlp-getting-started'
)
print('✅ Submission uploaded! Check score at:')
print('   https://www.kaggle.com/competitions/nlp-getting-started/submissions')

In [ ]:
# List all submissions and scores
submissions = kaggle.api.competition_submissions(competition='nlp-getting-started')

print(f'{'Ref':<12} {'Date':<22} {'Score':<10} {'Status':<12} {'Description'}')
print('-' * 75)
for s in submissions[:10]:   # show latest 10
    score = f"{s.publicScore:.5f}" if s.publicScore else 'N/A'
    print(f"{str(s.ref):<12} {str(s.date):<22} {score:<10} {s.status:<12} {s.description}")

---
## Section 13 — Docker Reference
Build and run commands for containerised deployment.

In [ ]:
# Write Dockerfile to disk
dockerfile = """\
FROM python:3.11-slim
WORKDIR /app
RUN pip install --no-cache-dir \\
    gliclass \\
    torch \\
    huggingface_hub \\
    wandb \\
    pandas \\
    flask
COPY run_model.py .
CMD ["python", "run_model.py"]
"""

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)

print('✅ Dockerfile written.')
print(dockerfile)

In [ ]:
# Docker build & run commands (reference — run in WSL terminal)
docker_commands = {
    'Build image': 
        'docker build -t gliclass-model .',

    'Run (basic)':
        'docker run --rm '
        '-v "/mnt/f/models/gliclass-llama-1.3B:/models/gliclass-llama-1.3B:ro" '
        'gliclass-model',

    'Run with W&B':
        'docker run --rm '
        '-v "/mnt/f/models/gliclass-llama-1.3B:/models/gliclass-llama-1.3B:ro" '
        '-v ~/.netrc:/root/.netrc:ro '
        'gliclass-model',

    'Run with Kaggle data + W&B':
        'docker run --rm '
        '-v "/mnt/f/models/gliclass-llama-1.3B:/models/gliclass-llama-1.3B:ro" '
        '-v "/mnt/f/datasets/disaster-tweets:/data" '
        '-v ~/.netrc:/root/.netrc:ro '
        'gliclass-model',

    'Run Flask web UI (http://localhost:8080)':
        'docker run --rm -p 0.0.0.0:8080:8080 '
        '-v "/mnt/f/models/gliclass-llama-1.3B:/models/gliclass-llama-1.3B:ro" '
        '-v ~/.netrc:/root/.netrc:ro '
        'gliclass-model',
}

for title, cmd in docker_commands.items():
    print(f"# {title}")
    print(f"  {cmd}")
    print()

In [ ]:
# Optionally run Docker build from notebook (requires Docker in WSL)
import subprocess

result = subprocess.run(
    ['docker', 'build', '-t', 'gliclass-model', '.'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else '')
if result.returncode == 0:
    print('✅ Docker image built: gliclass-model')
else:
    print('❌ Build failed:')
    print(result.stderr[-1000:])

---
## Section 14 — Common Errors & Fixes

| Error | Cause | Fix |
|---|---|---|
| `pip not found` | venv not activated | `source ~/mlops_venv/bin/activate` |
| `Operation not permitted` (venv) | NTFS drive blocks Linux perms | Create venv in `~/` not on F: |
| `No such file or directory` | Space in 'ML Ops' folder | Wrap path in quotes |
| `401 Unauthorized` (HF) | `HF_TOKEN` env var overrides login | `os.environ.pop('HF_TOKEN', None)` |
| `huggingface-cli deprecated` | Old CLI removed | Use `hf auth login` |
| `BFloat16 vs Float32 error` | Model weights dtype mismatch | Add `model = model.float()` |
| `docker: command not found` | WSL integration off | Docker Desktop → Settings → WSL Integration |
| All tweets = disaster | `threshold=0.3` too low | Use median threshold approach |
| All tweets = not disaster | `not_disaster` label scores higher | Use median — don't compare labels directly |
| `Kaggle 403 Forbidden` | Haven't accepted rules | Join competition at kaggle.com first |

---
## Section 15 — Final Summary

| Component | Status | Details |
|---|---|---|
| WSL Fedora Remix | ✅ | Windows WSL with Fedora Remix distro |
| VSCode + Jupyter | ✅ | WSL mode, Python (MLOps-A3) kernel |
| Virtual Environment | ✅ | `~/mlops_venv` — all packages installed |
| Hugging Face Login | ✅ | User: ausafahmad |
| Model Downloaded | ✅ | gliclass-llama-1.3B → /mnt/f/models/ |
| Docker WSL Integration | ✅ | Docker Desktop → FedoraRemix enabled |
| Docker Image Built | ✅ | `gliclass-model:latest` |
| Flask Web UI | ✅ | http://localhost:8080 |
| Kaggle Auth | ✅ | ausafiitj |
| W&B Connected | ✅ | Project: mlops-assignment3 |
| Kaggle v1 | ⚠️ 0.42966 | threshold=0.3 — all tweets = disaster |
| **Kaggle v3 BEST** | **✅ 0.48605** | **Median threshold — 1631/1632 split** |

In [ ]:
print("="*55)
print(" MLOps Assignment 3 — Complete")
print(" Ausaf Ahmad | MTech AI/ML | IIT Jodhpur")
print("="*55)
print()
print(" Best Kaggle Score : 0.48605")
print(" W&B Project       : mlops-assignment3")
print(" Model             : gliclass-llama-1.3B-v1.0")
print(" Strategy          : Median threshold zero-shot")
print()